# State Estimation with Transformers

This notebook is a brief demo on training a Transformer for a simple state estimation task. We will:

1. Create a synthetic state-estimation problem
2. Train a Transformer to predict future observations
3. Compare its performance against a Kalman filter

The settings here are intentionally small for a live run. A full run may use larger dimensions, wider/deeper Transformers, and many more training iterations.

## 0. Environment setup

This demo uses `uv` as its Python package manager. To install `uv`, please follow the instructions at https://docs.astral.sh/uv/getting-started/installation/

Run the next cell once from the repository root to create `./.venv` virtual environment, install the project dependencies, and register a Jupyter kernel. After it finishes, select the `Python (demo)` kernel.

The default command installs the CPU-compatible JAX package listed in `requirements.txt`. If your machine has an NVIDIA GPU, uncomment the GPU-enabled install line after checking the driver/CUDA constraints in the official JAX install guide: https://docs.jax.dev/en/latest/installation.html

In [ ]:
# Install dependencies
!uv venv
!uv pip install -r requirements.txt

# Optional NVIDIA GPU-enabled JAX installs.
# !uv pip install -U "jax[cuda13]"

# Add new environment to Jupyter
!uv run python -m ipykernel install --user --name demo --display-name "Python (demo)"

## 1. Imports and runtime check

The demo uses the following local modules:

- `task/filter.py`: synthetic state-estimation task and Kalman baseline
- `model/transformer.py`: Transformer configuration and model
- `train.py`: JAX/Flax training loop

In [ ]:
from pathlib import Path
import time

import jax
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from common import set_theme
from model.transformer import TransformerConfig
from task.filter import KalmanFilterTask, pred_kalman
from train import train

set_theme()

REPO_ROOT = Path.cwd()
print(f"repo: {REPO_ROOT}")
print(f"JAX devices: {jax.devices()}")

## 2. Define the state-estimation task

The state-estimation task is inspired by the classical Kalman filtering setup. The latent system is

$$x_t = A x_{t-1} + u_t, \qquad y_t = C x_t + w_t.$$

Each training example contains three pieces of context in one sequence:

1. Rows containing the transition matrix `A`,
2. Rows containing the observation matrix `C`, and
3. Rows containing the observed time series `y_t`.


In [ ]:
TASK_KWARGS = dict(
    length=16,
    n_dims=4,
    n_obs_dims=4,
    mode="ac",
    n_tasks=None,          # sample a new A for each example
    n_snaps=None,          # use the true A rather than a DMD estimate
    t_noise=1.0,
    o_noise=0.01,
    noise_dist="half",     # non-Gaussian process noise
    batch_size=128,
    seed=0,
)

train_task = KalmanFilterTask(**TASK_KWARGS)
sample = next(train_task)

n_context_rows = train_task.n_dims + train_task.n_obs_dims
print(f"batch shape: {sample.shape}")
print(f"context rows for A and C: {n_context_rows}")
print(f"observation rows: {train_task.length}")
print(f"feature dimension: {sample.shape[-1]}")

## 3. Configure and train the Transformer

The training loop performs autoregressive next-step prediction with squared error. This live demo uses a small model and reduced iterations to give a flavor for the training and results.

The main model knobs are width, number of layers, number of attention heads, positional embeddings, and training iterations.

In [ ]:
config = TransformerConfig(
    n_layers=2,
    n_hidden=128,
    n_heads=2,
    n_mlp_layers=2,
    pos_emb=False,
    layer_norm=True,
    residual_connections=True,
    return_final_logits_only=False,
    n_out=TASK_KWARGS["n_obs_dims"],
)

TRAIN_ITERS = 500
start = time.perf_counter()
state, hist = train(
    config,
    data_iter=iter(train_task),
    train_iters=TRAIN_ITERS,
    test_every=100,
    test_iters=1,
    seed=0,
)
elapsed = time.perf_counter() - start
print(f"training time: {elapsed:.1f} seconds")

In [ ]:
loss_df = pd.DataFrame(
    {
        "iteration": np.arange(100, TRAIN_ITERS + 1, 100),
        "train_loss": [float(m.loss) for m in hist["train"]],
        "test_loss": [float(m.loss) for m in hist["test"]],
    }
)

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(loss_df["iteration"], loss_df["train_loss"], "--o", label="Train", alpha=0.9, color="C0")
ax.plot(loss_df["iteration"], loss_df["test_loss"], "--o", label="Test", alpha=0.9, color="C1")
ax.set_xlabel("Training iteration")
ax.set_ylabel("Autoregressive MSE")
ax.legend().set_title(None)
sns.move_legend(ax, "lower left", bbox_to_anchor=(1, 0))
fig.tight_layout()

## 4. Evaluate against Kalman filtering

We now evaluate our Transformer against a Kalman filter. For evaluation, we extend the sequence length from 16 to 32 to also explore generalization beyond the training length.

The Kalman baseline receives the true transition and observation matrices. The zero predictor corresponds to a model that always predicts `0`, and function as a baseline for comparison.

In [ ]:
def split_ac_batch(xs, task):
    """Split a `mode='ac'` batch into A, C, and observation rows."""
    n_dims = task.n_dims
    n_obs_dims = task.n_obs_dims
    A = xs[:, :n_dims, n_obs_dims:]
    C = xs[:, n_dims:n_dims + n_obs_dims, n_obs_dims:]
    observations = xs[:, n_dims + n_obs_dims:, :n_obs_dims]
    return A, C, observations


def evaluate_model(state, task, eval_length=32, eval_batch_size=512):
    task.length = eval_length
    task.batch_size = eval_batch_size

    xs_full = next(task)
    A, C, observations = split_ac_batch(xs_full, task)

    transformer_pred = state.apply_fn({"params": state.params}, xs_full)[:, :-1]
    transformer_mse = ((xs_full[:, 1:, :task.n_obs_dims] - transformer_pred) ** 2).mean(axis=(0, -1))
    transformer_mse = np.asarray(transformer_mse)[task.n_dims + task.n_obs_dims:]

    kalman_pred, kalman_est_var = pred_kalman(
        observations,
        task,
        A=A,
        C=C,
        t_noise=task.t_noise,
        o_noise=task.o_noise,
    )
    kalman_mse = ((observations[:, 1:] - kalman_pred[:, :-1]) ** 2).mean(axis=(0, -1))
    zero_mse = (observations[:, 1:] ** 2).mean(axis=(0, -1))

    n = min(len(transformer_mse), len(kalman_mse), len(zero_mse), len(kalman_est_var))
    return pd.DataFrame(
        {
            "time": np.arange(n),
            "Transformer (NoPE)": transformer_mse[:n],
            "Kalman Filter": kalman_mse[:n],
            "Kalman Est. Var.": np.asarray(kalman_est_var)[:n],
            "Zero Predictor": zero_mse[:n],
        }
    )


eval_df = evaluate_model(state, train_task)
eval_df.tail()

In [ ]:
start_idx = 1
train_test_boundary = TASK_KWARGS["length"] - 2

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(
    eval_df["time"][start_idx:],
    eval_df["Transformer (NoPE)"][start_idx:],
    "--o",
    label="Transformer (NoPE)",
    alpha=0.7,
    color="C9",
)
ax.plot(
    eval_df["time"][start_idx:],
    eval_df["Kalman Filter"][start_idx:],
    "--o",
    label="Kalman Filter",
    alpha=0.7,
    color="C1",
)

ax.plot(
    eval_df["time"][start_idx:],
    eval_df["Zero Predictor"][start_idx:],
    "--o",
    label="Zero Predictor",
    alpha=0.7,
    color="C8",
)

ax.axvline(x=train_test_boundary, linestyle="dashed", color="gray", alpha=0.9)
ymin, ymax = ax.get_ylim()
ax.text(
    train_test_boundary - 0.5,
    ymin + 0.65 * (ymax - ymin),
    "train/test",
    rotation=90,
    va="center",
    color="gray",
    fontsize=9,
)

ax.set_xlabel("Time")
ax.set_ylabel("MSE")
ax.legend().set_title(None)
sns.move_legend(ax, "lower left", bbox_to_anchor=(1, 0))
fig.tight_layout()

## 5. Adapting to other tasks

The following components are important consider when adapting this code to other tasks:

- **Task/data interface:** update or replace `KalmanFilterTask` so batches encode the user's observations and any known system context.
- **Context convention:** decide what matrices, metadata, or calibration observations are prepended before the time series.
- **Model configuration:** tune `TransformerConfig` for the observation dimension, sequence length, context length, width, heads, layers, and positional encoding choice.
- **Training schedule:** increase `train_iters`, batch size, and model size for production-quality results.
- **Evaluation:** define the prediction horizon, distribution shift tests, and domain-specific error metric.